In [1]:
import numpy as np
import pandas as pd
import os

DATA_DIR = r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\raw"

orders = pd.read_csv(os.path.join(DATA_DIR, "olist_orders_dataset.csv"))
items = pd.read_csv(os.path.join(DATA_DIR, "olist_order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(DATA_DIR, "olist_customers_dataset.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(DATA_DIR, "olist_sellers_dataset.csv"))
payments= pd.read_csv(os.path.join(DATA_DIR, "olist_order_payments_dataset.csv"))
geolocation = pd.read_csv(os.path.join(DATA_DIR, "olist_geolocation_dataset.csv"))
category_translation = pd.read_csv(os.path.join(DATA_DIR, "product_category_name_translation.csv"))
reviews = pd.read_csv(os.path.join(DATA_DIR, "olist_order_reviews_dataset.csv"))


tables = {
    'orders': orders, 'items': items, 'customers': customers,
    'products': products, 'sellers': sellers, 'payments': payments,
    'reviews': reviews, 'geolocation': geolocation,
    'category_translation': category_translation
}

In [2]:
for name, df in tables.items():
    print(f"\n{'='*50}\n{name.upper()} — shape: {df.shape}")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum()[df.isna().sum() > 0])
    print(f"Duplicate rows: {df.duplicated().sum()}")


ORDERS — shape: (99441, 8)
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
Duplicate rows: 0

ITEMS — shape: (112650, 7)
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Missing values:
Series([], dtype: int64)
Duplicate rows: 0

CUSTOMERS — shape: (99441, 5)
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state

In [3]:
expected = {'orders': 99441, 'items': 112650, 'customers': 99441,
            'sellers': 3095, 'payments': 103886}
for name, count in expected.items():
    actual = len(tables[name])
    status = "OK" if actual == count else "MISMATCH"
    print(f"{name}: expected {count}, got {actual} — {status}")

orders: expected 99441, got 99441 — OK
items: expected 112650, got 112650 — OK
customers: expected 99441, got 99441 — OK
sellers: expected 3095, got 3095 — OK
payments: expected 103886, got 103886 — OK


In [4]:
assert orders['order_id'].is_unique
assert customers['customer_id'].is_unique
assert products['product_id'].is_unique
assert sellers['seller_id'].is_unique
print("All primary keys confirmed unique.")

All primary keys confirmed unique.


In [5]:
orphan_items = ~items['order_id'].isin(orders['order_id'])
print(f"Orphan order_items: {orphan_items.sum()}")

Orphan order_items: 0


In [21]:
orphan_orders = ~orders['customer_id'].isin(customers['customer_id'])
print(f"Orphan orders (bad customer_id): {orphan_orders.sum()}")

Orphan orders (bad customer_id): 0


In [6]:
orphan_products = ~items['product_id'].isin(products['product_id'])
print(f"Orphan items (bad product_id): {orphan_products.sum()}")

Orphan items (bad product_id): 0


In [7]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
orders[date_cols] = orders[date_cols].apply(pd.to_datetime, errors='coerce')

bad_order = orders[orders['order_purchase_timestamp'] > orders['order_delivered_customer_date']]
print(f"Orders with delivery before purchase (should be ~0): {len(bad_order)}")

Orders with delivery before purchase (should be ~0): 0


In [8]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
orders[date_cols] = orders[date_cols].apply(pd.to_datetime, errors='coerce')

bad_order = orders[orders['order_purchase_timestamp'] > orders['order_delivered_customer_date']]
print(f"Orders with delivery before purchase (should be ~0): {len(bad_order)}")

Orders with delivery before purchase (should be ~0): 0


In [9]:
orders['is_delivered'] = orders['order_delivered_customer_date'].notna()

before = len(orders)
orders = orders[~((orders['order_status'] == 'canceled') &
                   orders['order_delivered_customer_date'].isna())]
print(f"Dropped {before - len(orders)} cancelled/undelivered rows ({(before-len(orders))/before:.1%})")

Dropped 619 cancelled/undelivered rows (0.6%)


In [5]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [10]:
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('no title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('no comment')

before = len(reviews)
reviews = reviews.drop_duplicates()
print(f"Dropped {before - len(reviews)} duplicate review rows")

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

Dropped 0 duplicate review rows


In [11]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [27]:
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm',
            'product_width_cm', 'product_name_lenght', 'product_description_lenght',
            'product_photos_qty']
print(products[dim_cols].isna().sum())

product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64


In [12]:
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[col] = products.groupby('product_category_name')[col] \
        .transform(lambda x: x.fillna(x.median()))

before = len(products)
products = products.drop_duplicates(subset='product_id')
print(f"Dropped {before - len(products)} duplicate products")

Dropped 0 duplicate products


In [13]:
items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [14]:
before = len(items)
items = items.drop_duplicates(subset=['order_id', 'order_item_id'])
print(f"Dropped {before - len(items)} duplicate item rows")

Dropped 0 duplicate item rows


In [15]:
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

print((items['price'] <= 0).sum(), "rows with price <= 0")
print((items['freight_value'] < 0).sum(), "rows with negative freight")

0 rows with price <= 0
0 rows with negative freight


In [14]:
customers.isna().sum() 

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [16]:
before = len(customers)
customers = customers.drop_duplicates(subset='customer_id')
print(f"Dropped {before - len(customers)} duplicate customers")

customers['customer_state'] = customers['customer_state'].str.upper().str.strip()
customers['customer_city'] = customers['customer_city'].str.strip().str.lower()

Dropped 0 duplicate customers


In [16]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [32]:
before = len(sellers)
sellers = sellers.drop_duplicates(subset='seller_id')
print(f"Dropped {before - len(sellers)} duplicate sellers")

sellers['seller_state'] = sellers['seller_state'].str.upper().str.strip()
sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower()

Dropped 0 duplicate sellers


In [18]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [33]:
before = len(payments)
payments = payments.drop_duplicates()
print(f"Dropped {before - len(payments)} duplicate payment rows")

zero_pay = (payments['payment_value'] == 0).sum()
print(f"Zero-value payment rows: {zero_pay}")

payments['payment_type'] = payments['payment_type'].str.strip().str.lower()

Dropped 0 duplicate payment rows
Zero-value payment rows: 9


In [34]:
geolocation.isna().sum()
print(f"Total rows: {len(geolocation)}, unique zip prefixes: {geolocation['geolocation_zip_code_prefix'].nunique()}")

Total rows: 1000163, unique zip prefixes: 19015


In [35]:
geo_clean = geolocation.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
).reset_index()

print(f"Collapsed from {len(geolocation)} to {len(geo_clean)} rows")

Collapsed from 1000163 to 19015 rows


In [18]:
checks = {
    'items → orders': ~items['order_id'].isin(orders['order_id']),
    'items → products': ~items['product_id'].isin(products['product_id']),
    'items → sellers': ~items['seller_id'].isin(sellers['seller_id']),
    'orders → customers': ~orders['customer_id'].isin(customers['customer_id']),
    'payments → orders': ~payments['order_id'].isin(orders['order_id']),
    'reviews → orders': ~reviews['order_id'].isin(orders['order_id']),
}

for label, mask in checks.items():
    print(f"{label}: {mask.sum()} orphan rows")

items → orders: 535 orphan rows
items → products: 0 orphan rows
items → sellers: 0 orphan rows
orders → customers: 0 orphan rows
payments → orders: 658 orphan rows
reviews → orders: 603 orphan rows


In [19]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products = products.merge(category_translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

In [20]:
id_cols = ['order_id', 'customer_id', 'product_id', 'seller_id']
for df in [orders, items, customers, products, sellers]:
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)

In [21]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

orders[date_cols] = orders[date_cols].apply(pd.to_datetime, errors='coerce')

# now check
print(orders[date_cols].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [22]:
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] -
                                   orders['order_estimated_delivery_date']).dt.days
orders['is_late'] = orders['delivery_delay_days'] > 0

items_agg = items.groupby('order_id').agg(
    order_value=('price', 'sum'),
    freight_value=('freight_value', 'sum')
).reset_index()

fact_orders = orders.merge(items_agg, on='order_id', how='left') \
                     .merge(customers[['customer_id', 'customer_state', 'customer_city']],
                            on='customer_id', how='left')
fact_orders['order_total'] = fact_orders['order_value'] + fact_orders['freight_value']

In [23]:
python_total = fact_orders['order_total'].sum()
payments_total = payments['payment_value'].sum()
diff_pct = abs(python_total - payments_total) / payments_total
print(f"Order totals: {python_total:.2f} | Payments total: {payments_total:.2f} | Diff: {diff_pct:.2%}")

Order totals: 15738416.10 | Payments total: 16008872.12 | Diff: 1.69%


In [24]:
import os
os.makedirs('data/processed', exist_ok=True)

orders.to_csv('data/processed/orders_clean.csv', index=False)
items.to_csv('data/processed/items_clean.csv', index=False)
customers.to_csv('data/processed/customers_clean.csv', index=False)
products.to_csv('data/processed/products_clean.csv', index=False)
sellers.to_csv('data/processed/sellers_clean.csv', index=False)
payments.to_csv('data/processed/payments_clean.csv', index=False)
reviews.to_csv('data/processed/reviews_clean.csv', index=False)
geo_clean.to_csv('data/processed/geolocation_clean.csv', index=False)

NameError: name 'geo_clean' is not defined

In [29]:
import os

BASE_DIR = r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Saving to: {PROCESSED_DIR}")

Saving to: C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed


In [30]:
# --- Build fact_orders (orders + item aggregates + customer location + totals) ---
items_agg = items.groupby('order_id').agg(
    order_value=('price', 'sum'),
    freight_value=('freight_value', 'sum')
).reset_index()

fact_orders = orders.merge(items_agg, on='order_id', how='left') \
                     .merge(customers[['customer_id', 'customer_state', 'customer_city']],
                            on='customer_id', how='left')

fact_orders['order_total'] = fact_orders['order_value'] + fact_orders['freight_value']

# Sanity check before export
print(fact_orders[['order_id', 'order_value', 'freight_value', 'order_total',
                    'delivery_delay_days', 'is_late']].head())
print(f"fact_orders shape: {fact_orders.shape}")

# --- fact_order_items (item-level detail: product, seller, price) ---
fact_order_items = items[['order_id', 'order_item_id', 'product_id', 'seller_id',
                           'price', 'freight_value']]

# --- dim_products ---
dim_products = products[['product_id', 'product_category_name_english',
                          'product_weight_g', 'product_length_cm',
                          'product_height_cm', 'product_width_cm']]

# --- dim_customers ---
dim_customers = customers[['customer_id', 'customer_unique_id',
                            'customer_state', 'customer_city']]

# --- dim_sellers ---
dim_sellers = sellers[['seller_id', 'seller_state', 'seller_city']]

# --- fact_payments ---
fact_payments = payments[['order_id', 'payment_sequential', 'payment_type',
                           'payment_installments', 'payment_value']]

# --- fact_reviews ---
fact_reviews = reviews[['review_id', 'order_id', 'review_score',
                         'review_creation_date', 'review_answer_timestamp']]

                           order_id  order_value  freight_value  order_total  \
0  e481f51cbdc54678b7cc49136f2d6af7        29.99           8.72        38.71   
1  53cdb2fc8bc7dce0b6741e2150273451       118.70          22.76       141.46   
2  47770eb9100c2d0c44946d9cf07ec65d       159.90          19.22       179.12   
3  949d5b44dbf5de918fe9c16f97b45f8a        45.00          27.20        72.20   
4  ad21c59c0840e6cb83a9ceb5573f8159        19.90           8.72        28.62   

   delivery_delay_days  is_late  
0                 -8.0    False  
1                 -6.0    False  
2                -18.0    False  
3                -13.0    False  
4                -10.0    False  
fact_orders shape: (98822, 16)


In [31]:
fact_orders.to_csv(os.path.join(PROCESSED_DIR, "fact_orders.csv"), index=False)
fact_order_items.to_csv(os.path.join(PROCESSED_DIR, "fact_order_items.csv"), index=False)
dim_products.to_csv(os.path.join(PROCESSED_DIR, "dim_products.csv"), index=False)
dim_customers.to_csv(os.path.join(PROCESSED_DIR, "dim_customers.csv"), index=False)
dim_sellers.to_csv(os.path.join(PROCESSED_DIR, "dim_sellers.csv"), index=False)
fact_payments.to_csv(os.path.join(PROCESSED_DIR, "fact_payments.csv"), index=False)
fact_reviews.to_csv(os.path.join(PROCESSED_DIR, "fact_reviews.csv"), index=False)

print("All fact/dim tables exported to:", PROCESSED_DIR)
print(os.listdir(PROCESSED_DIR))

All fact/dim tables exported to: C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed
['dim_customers.csv', 'dim_products.csv', 'dim_sellers.csv', 'fact_orders.csv', 'fact_order_items.csv', 'fact_payments.csv', 'fact_reviews.csv']


In [32]:
import pandas as pd
df = pd.read_csv(r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed\fact_orders.csv", nrows=0)
print(df.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_delivered', 'delivery_delay_days', 'is_late', 'order_value', 'freight_value', 'customer_state', 'customer_city', 'order_total']


In [33]:
orphan_items = items[~items['order_id'].isin(orders['order_id'])]
print(f"Orphan order_items: {len(orphan_items)} out of {len(items)}")
print(f"That's {len(orphan_items)/len(items):.2%} of item rows")

Orphan order_items: 535 out of 112650
That's 0.47% of item rows


In [34]:
fact_order_items_clean = fact_order_items[fact_order_items['order_id'].isin(fact_orders['order_id'])]

print(f"Dropped {len(fact_order_items) - len(fact_order_items_clean)} orphan item rows")

fact_order_items_clean.to_csv(
    r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed\fact_order_items.csv",
    index=False
)

Dropped 535 orphan item rows


In [35]:
fact_payments_clean = fact_payments[fact_payments['order_id'].isin(fact_orders['order_id'])]

print(f"Dropped {len(fact_payments) - len(fact_payments_clean)} orphan item rows")

fact_payments_clean.to_csv(
      r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed\fact_payments.csv",
    index=False
)
    

Dropped 658 orphan item rows


In [36]:
PROCESSED_DIR = r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed"
fact_reviews = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_reviews.csv"))

dupe_ids = fact_reviews[fact_reviews.duplicated(subset='review_id', keep=False)]
print(f"Rows involved in review_id duplicates: {len(dupe_ids)}")
dupe_ids.sort_values('review_id').head(20)

Rows involved in review_id duplicates: 1603


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,2017-09-09 00:00:00,2017-09-13 09:52:44


In [37]:
before = len(fact_reviews)
fact_reviews_clean = fact_reviews.drop_duplicates(subset='review_id', keep='first')
print(f"Dropped {before - len(fact_reviews_clean)} duplicate review_id rows")

fact_reviews_clean.to_csv(os.path.join(PROCESSED_DIR, "fact_reviews.csv"), index=False)

Dropped 814 duplicate review_id rows


In [39]:

PROCESSED_DIR = r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed"

fact_orders = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_orders.csv"))
fact_reviews = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_reviews.csv"))

In [40]:
orphan_reviews = fact_reviews[~fact_reviews['order_id'].isin(fact_orders['order_id'])]
print(f"Orphan reviews: {len(orphan_reviews)} out of {len(fact_reviews)}")

before = len(fact_reviews)
fact_reviews_clean = fact_reviews[fact_reviews['order_id'].isin(fact_orders['order_id'])]
print(f"Dropped {before - len(fact_reviews_clean)} orphan review rows")

fact_reviews_clean.to_csv(os.path.join(PROCESSED_DIR, "fact_reviews.csv"), index=False)

Orphan reviews: 571 out of 98410
Dropped 571 orphan review rows
